# ============================================
#  Notebook 11 — HCAT Embedding-Based Evaluation
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 11: HCAT Embedding Evaluation Metrics

**Purpose:**  
Run the `EmbeddingEvaluator` (HCAT framework) on per-case clinical text to measure  
the four HCAT embedding dimensions for each of the 14 clinical features.

**Two evaluation modes (both run in this notebook):**

| Mode | When | What is measured |
|------|------|------------------|
| **Context Quality** | Now (no LLM outputs needed) | Does filtered source text answer the feature query? |
| **Answer Quality** | After NB09 | Are LLM-extracted values grounded and complete? |

**HCAT Dimensions:**
- **Context Relevancy** — Does the retrieved section answer the feature query?
- **Groundedness** — Is the LLM answer grounded in the provided source text?
- **Completeness** — Does the answer cover all key phrases from the reference?
- **Answer Relevancy** — Does the answer actually address the extraction question?

**Inputs:**
- `DATA_PRIVATE_DIR/extracted_text_consolidated/*.txt` — per-case consolidated text (NB04b)
- `data/processed/rag_question_dataset.csv` — feature queries per case (NB04c)
- `DATA_PRIVATE_DIR/raw/merged_llm_summary_validation_datasheet*.xlsx` — ground truth labels
- `data/processed/nb09_llm_extraction_results.csv` *(optional)* — LLM answers from NB09

**Outputs:**
- `data/processed/hcat_context_quality.csv` — context-mode embedding metrics per (case, feature)
- `data/processed/hcat_answer_quality.csv` — answer-mode embedding metrics (when NB09 available)
- `reports/hcat_embedding_summary.png` — stratified metric plots
- `reports/hcat_correlation_with_labels.png` — embedding metric vs validation label agreement

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    "/Users/robertjames/Documents/GitHub/llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", "/Users/robertjames/data_private"))

CONSOLIDATED_DIR = DATA_PRIVATE_DIR / "extracted_text_consolidated"
PROCESSED_DIR    = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR       = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Module paths ──────────────────────────────────────────────────────────────
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "modeling"))  # for bare imports in hcat_evaluation_pipeline

from llm_eval_by_llm.feature_document_context import (
    FEATURE_DOCUMENT_CONTEXT,
    EXTRACTION_HINTS,
    FEATURE_SECTIONS,
    HIGH_FABRICATION_RISK,
    filter_text_to_relevant_sections,
)
from modeling.embedding_evaluation_metrics import (
    EmbeddingEvaluator,
    EmbeddingMetricsReport,
    create_metrics_summary_df,
)

SECTION_HEADER_MAP = {
    "hpi":       "=== HPI / CLINICAL NOTES ===",
    "radiology": "=== RADIOLOGY REPORTS ===",
    "pathology": "=== PATHOLOGY REPORTS ===",
    "genetics":  "=== GENETICS / MOLECULAR ===",
}

print(f"Features loaded  : {len(FEATURE_DOCUMENT_CONTEXT)}")
print(f"Consolidated dir : {CONSOLIDATED_DIR}")
consolidated_files = sorted(CONSOLIDATED_DIR.glob("*.txt"))
print(f"Case files found : {len(consolidated_files)}")

---
## Part 1: Load Data Sources

In [ ]:
# ── Validation datasheet (ground truth) ───────────────────────────────────────
DEID_PATH = DATA_PRIVATE_DIR / "deidentified" / "validation_datasheet_deidentified.xlsx"
RAW_PATH  = DATA_PRIVATE_DIR / "raw" / "merged_llm_summary_validation_datasheet.xlsx"
DATA_PATH = DEID_PATH if DEID_PATH.exists() else RAW_PATH

val_df = pd.read_excel(DATA_PATH)
val_df = val_df.replace(["NA", "na", "N/A", "n/a"], pd.NA)
print(f"Validation sheet : {val_df.shape[0]} obs × {val_df.shape[1]} cols")

# ── RAG question dataset (from NB04c) ─────────────────────────────────────────
Q_PATH = PROCESSED_DIR / "rag_question_dataset.csv"
if Q_PATH.exists():
    df_questions = pd.read_csv(Q_PATH)
    print(f"RAG questions    : {len(df_questions)} rows")
    print(f"  Case-specific  : {(df_questions['question_type']=='case_specific').sum()}")
    print(f"  Template       : {(df_questions['question_type']=='template').sum()}")
else:
    df_questions = None
    print("RAG question dataset not found — run NB04c first. Using extraction hints as queries.")

# ── Consolidated case text ────────────────────────────────────────────────────
corpus = {}
for f in consolidated_files:
    corpus[f.stem] = f.read_text(encoding="utf-8")
print(f"Corpus cases     : {len(corpus)}")

# ── NB09 LLM outputs (optional) ───────────────────────────────────────────────
NB09_PATH = PROCESSED_DIR / "nb09_llm_extraction_results.csv"
if NB09_PATH.exists():
    df_llm = pd.read_csv(NB09_PATH)
    print(f"LLM extraction results: {len(df_llm)} rows")
    ANSWER_MODE_AVAILABLE = True
else:
    df_llm = None
    ANSWER_MODE_AVAILABLE = False
    print("NB09 outputs not found — answer-quality mode will be skipped.")

In [ ]:
# ── Build query lookup: feature_key → query string ───────────────────────────
# Prefer case-specific questions from NB04c; fall back to extraction hints

def get_query_for_case_feature(
    patient_case_id: str,
    feature_key: str,
    df_questions: pd.DataFrame = None
) -> str:
    """Return best available question for this (case, feature) pair."""
    if df_questions is not None:
        # Case-specific question first
        match = df_questions[
            (df_questions["patient_case_id"] == patient_case_id) &
            (df_questions["feature_key"] == feature_key) &
            (df_questions["question_type"] == "case_specific")
        ]
        if len(match) > 0:
            return match.iloc[0]["question"]
        # Template fallback
        tmpl = df_questions[
            (df_questions["feature_key"] == feature_key) &
            (df_questions["question_type"] == "template")
        ]
        if len(tmpl) > 0:
            return tmpl.iloc[0]["question"]
    # Last resort: extraction hint as query
    return FEATURE_DOCUMENT_CONTEXT[feature_key]["extraction_hint"]

# Spot check
sample_feature = "lesion_size"
sample_case = list(corpus.keys())[0] if corpus else "CASE_TEST"
print(f"Sample query for {sample_feature} / {sample_case}:")
print(f"  {get_query_for_case_feature(sample_case, sample_feature, df_questions)[:200]}")

---
## Part 2: Initialise Embedding Evaluator

Uses `all-mpnet-base-v2` (768-dim, same model as NB05 BERT embeddings).  
Falls back to character-trigram vectors if `sentence-transformers` is not installed.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print("sentence-transformers available")
except ImportError:
    ST_AVAILABLE = False
    print("sentence-transformers not installed — using character-trigram fallback")
    print("Install: pip install sentence-transformers")

EMBEDDING_MODEL = "all-mpnet-base-v2"
evaluator = EmbeddingEvaluator(model_name=EMBEDDING_MODEL)
print(f"EmbeddingEvaluator ready (model={EMBEDDING_MODEL})")

# Smoke test
smoke = evaluator.evaluate(
    query="What is the lesion size?",
    answer="The lesion measures 1.2 cm in greatest dimension.",
    reference="Lesion size: 12 mm (1.2 cm) on ultrasound.",
    source_documents=["Ultrasound report: 1.2 cm mass in left breast upper outer quadrant."]
)
print(f"  Context Relevancy : {smoke.context_relevancy:.3f}")
print(f"  Groundedness      : {smoke.groundedness:.3f}")
print(f"  Completeness      : {smoke.completeness:.3f}")
print(f"  Answer Relevancy  : {smoke.answer_relevancy:.3f}")

---
## Part 3: Context Quality Mode

**No LLM outputs required.** For each (case, feature):  
- `query` = feature extraction question  
- `answer` = filtered consolidated text for the relevant sections (what the LLM *would* see)  
- `source_documents` = same filtered text split by document sub-headers  

Measures: **does the source document context contain enough information to answer this feature query?**  
Low context quality → LLM is more likely to fabricate.

In [ ]:
def split_into_source_docs(filtered_text: str) -> list:
    """Split consolidated text at document sub-headers (--- filename ---) into separate doc strings."""
    import re
    parts = re.split(r'--- .+? ---', filtered_text)
    return [p.strip() for p in parts if p.strip()]

context_rows = []

if not corpus:
    print("No consolidated case files found — run NB04 + NB04b first.")
else:
    for patient_case_id, full_text in tqdm(corpus.items(), desc="Context quality"):
        for feature_key, ctx in FEATURE_DOCUMENT_CONTEXT.items():

            # Filter text to sections relevant to this feature
            filtered = filter_text_to_relevant_sections(
                full_text, feature_key,
                section_header_map=SECTION_HEADER_MAP
            )

            if not filtered.strip():
                # No relevant sections in this case — skip
                context_rows.append({
                    "patient_case_id":  patient_case_id,
                    "feature_key":      feature_key,
                    "feature_display":  ctx["display"],
                    "domain":           ctx["domain"],
                    "fabrication_risk": ctx["fabrication_risk"],
                    "primary_sections": "|".join(ctx["primary_sections"]),
                    "context_chars":    0,
                    "n_source_docs":    0,
                    "context_relevancy": np.nan,
                    "groundedness":      np.nan,
                    "completeness":      np.nan,
                    "answer_relevancy":  np.nan,
                    "semantic_similarity": np.nan,
                    "token_overlap_f1": np.nan,
                    "key_phrase_coverage": np.nan,
                    "requires_human_review": None,
                    "low_confidence_dims": "",
                    "mode": "context",
                })
                continue

            query = get_query_for_case_feature(patient_case_id, feature_key, df_questions)
            source_docs = split_into_source_docs(filtered)

            try:
                report = evaluator.evaluate(
                    query=query,
                    answer=filtered[:3000],          # proxy: context as answer
                    reference=filtered[:3000],        # self-reference for context quality
                    retrieved_documents=source_docs,  # individual docs
                    source_documents=source_docs,
                )
            except Exception as e:
                print(f"  Error {patient_case_id}/{feature_key}: {e}")
                continue

            context_rows.append({
                "patient_case_id":     patient_case_id,
                "feature_key":         feature_key,
                "feature_display":     ctx["display"],
                "domain":              ctx["domain"],
                "fabrication_risk":    ctx["fabrication_risk"],
                "primary_sections":    "|".join(ctx["primary_sections"]),
                "context_chars":       len(filtered),
                "n_source_docs":       len(source_docs),
                "context_relevancy":   report.context_relevancy,
                "groundedness":        report.groundedness,
                "completeness":        report.completeness,
                "answer_relevancy":    report.answer_relevancy,
                "semantic_similarity": report.semantic_similarity,
                "token_overlap_f1":    report.token_overlap_f1,
                "key_phrase_coverage": report.key_phrase_coverage,
                "requires_human_review": report.requires_human_review,
                "low_confidence_dims":   "|".join(report.low_confidence_dimensions),
                "mode": "context",
            })

    df_context = pd.DataFrame(context_rows)
    df_context.to_csv(PROCESSED_DIR / "hcat_context_quality.csv", index=False)
    n_ok = df_context["context_relevancy"].notna().sum()
    print(f"\nContext quality evaluated: {n_ok} / {len(df_context)} (case × feature) pairs")
    print(f"Saved → {PROCESSED_DIR / 'hcat_context_quality.csv'}")
    df_context.head(4)

---
## Part 4: Answer Quality Mode (requires NB09 outputs)

For each (case, feature, model, prompt):  
- `query` = feature extraction question  
- `answer` = LLM-extracted text from NB09  
- `reference` = ground truth label text (expected values from source)  
- `source_documents` = filtered consolidated text sections  

Measures: **how grounded, complete and relevant is the actual LLM answer?**

In [ ]:
# Build ground truth reference text per (case, feature) from source labels
# Uses the validation datasheet source columns + expected_values from feature context

SOURCE_COL_MAP = {
    "lesion_size":                        "lesion_size_status_source",
    "laterality":                         "laterality_status_source",
    "lesion_location":                    "lesion_location_status_source",
    "calcifications_asymmetry":           "calcifications_asymmetry_status_source",
    "additional_enhancement_mri":         "additional_enhancement_mri_status_source",
    "extent":                             "extent_status_source",
    "accurate_clip_placement":            "accurate_clip_placement_status_source",
    "workup_recommendation":              "workup_recommendation_status_source",
    "lymph_node":                         "Lymph node_status_source",
    "chronology_preserved":               "chronology_preserved_status_source",
    "biopsy_method":                      "biopsy_method_status_source",
    "invasive_component_size_pathology":  "invasive_component_size_pathology_status_source",
    "histologic_diagnosis":               "histologic_diagnosis_status_source",
    "receptor_status":                    "receptor_status_source",
}

def build_reference_text(feature_key: str, source_val) -> str:
    """Build a short reference string from source label + expected values."""
    ctx = FEATURE_DOCUMENT_CONTEXT[feature_key]
    present = int(source_val) == 1 if pd.notna(source_val) else None
    if present is None:
        return ""
    status = "present" if present else "absent/not applicable"
    return f"{ctx['display']} is {status}. Expected: {ctx['expected_values'][:200]}"

print("Reference builder ready.")
print("Sample:", build_reference_text("lesion_size", 1))

In [ ]:
if not ANSWER_MODE_AVAILABLE:
    print("Answer-quality mode skipped — run NB09 first to generate LLM extraction results.")
    print(f"Expected path: {NB09_PATH}")
    df_answer = pd.DataFrame()
else:
    # NB09 output schema: patient_case_id, feature_key, model, prompt_id, extracted_answer
    required_cols = {"patient_case_id", "feature_key", "extracted_answer"}
    missing = required_cols - set(df_llm.columns)
    if missing:
        print(f"NB09 output missing columns: {missing}")
        df_answer = pd.DataFrame()
    else:
        answer_rows = []
        for _, row in tqdm(df_llm.iterrows(), total=len(df_llm), desc="Answer quality"):
            patient_case_id = row["patient_case_id"]
            feature_key     = row["feature_key"]
            answer_text     = str(row["extracted_answer"])

            if feature_key not in FEATURE_DOCUMENT_CONTEXT:
                continue
            if patient_case_id not in corpus:
                continue

            ctx = FEATURE_DOCUMENT_CONTEXT[feature_key]
            query  = get_query_for_case_feature(patient_case_id, feature_key, df_questions)
            filtered = filter_text_to_relevant_sections(
                corpus[patient_case_id], feature_key,
                section_header_map=SECTION_HEADER_MAP
            )
            source_docs = split_into_source_docs(filtered)

            # Get source label for reference construction
            source_col = SOURCE_COL_MAP.get(feature_key)
            source_val = None
            if source_col and source_col in val_df.columns:
                # Try to match by position (obs index) — adjust if you have a case_id join
                obs_match = df_llm.index[df_llm["patient_case_id"] == patient_case_id].tolist()
                if obs_match and obs_match[0] < len(val_df):
                    source_val = val_df.iloc[obs_match[0]][source_col]
            reference = build_reference_text(feature_key, source_val)

            try:
                report = evaluator.evaluate(
                    query=query,
                    answer=answer_text,
                    reference=reference if reference else None,
                    source_documents=source_docs if source_docs else None,
                    retrieved_documents=source_docs if source_docs else None,
                )
            except Exception as e:
                print(f"  Error {patient_case_id}/{feature_key}: {e}")
                continue

            answer_rows.append({
                "patient_case_id":     patient_case_id,
                "feature_key":         feature_key,
                "feature_display":     ctx["display"],
                "domain":              ctx["domain"],
                "fabrication_risk":    ctx["fabrication_risk"],
                "model":               row.get("model", "unknown"),
                "prompt_id":           row.get("prompt_id", "unknown"),
                "approach":            row.get("approach", "unknown"),
                "context_relevancy":   report.context_relevancy,
                "groundedness":        report.groundedness,
                "completeness":        report.completeness,
                "answer_relevancy":    report.answer_relevancy,
                "semantic_similarity": report.semantic_similarity,
                "token_overlap_f1":    report.token_overlap_f1,
                "key_phrase_coverage": report.key_phrase_coverage,
                "requires_human_review": report.requires_human_review,
                "low_confidence_dims": "|".join(report.low_confidence_dimensions),
                "mode": "answer",
            })

        df_answer = pd.DataFrame(answer_rows)
        df_answer.to_csv(PROCESSED_DIR / "hcat_answer_quality.csv", index=False)
        print(f"\nAnswer quality evaluated: {len(df_answer)} rows")
        print(f"Saved → {PROCESSED_DIR / 'hcat_answer_quality.csv'}")

---
## Part 5: Stratified Metrics — by Feature, Domain, Fabrication Risk

In [ ]:
# Work with context quality results (always available)
ok = df_context[df_context["context_relevancy"].notna()].copy()
METRIC_COLS = ["context_relevancy", "groundedness", "completeness", "answer_relevancy"]

print(f"Cases with content : {ok['patient_case_id'].nunique()}")
print(f"Total eval pairs   : {len(ok)}")
print()

# ── Overall means ─────────────────────────────────────────────────────────────
print("Overall HCAT metrics (context quality):")
for m in METRIC_COLS:
    print(f"  {m:<25} : {ok[m].mean():.3f} ± {ok[m].std():.3f}")

# ── Per-feature ───────────────────────────────────────────────────────────────
feature_agg = (
    ok.groupby(["feature_display", "domain", "fabrication_risk"])[METRIC_COLS]
    .mean()
    .round(3)
    .reset_index()
    .sort_values("context_relevancy", ascending=False)
)
print("\nPer-feature context relevancy (top and bottom):")
print(feature_agg[["feature_display", "domain", "fabrication_risk", "context_relevancy", "groundedness"]]
      .to_string(index=False))

# ── By domain ─────────────────────────────────────────────────────────────────
domain_agg = ok.groupby("domain")[METRIC_COLS].mean().round(3)
print("\nBy domain:")
print(domain_agg.to_string())

# ── By fabrication risk ───────────────────────────────────────────────────────
fab_agg = ok.groupby("fabrication_risk")[METRIC_COLS].mean().round(3)
print("\nBy fabrication risk:")
print(fab_agg.to_string())

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
FAB_COLORS = {"high": "#e74c3c", "medium": "#f39c12", "low": "#2ecc71"}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("HCAT Embedding Evaluation — Context Quality Mode",
             fontsize=16, fontweight="bold")

# 1. Context relevancy by feature
fa = feature_agg.sort_values("context_relevancy")
cols1 = [FAB_COLORS.get(r, "#95a5a6") for r in fa["fabrication_risk"]]
axes[0, 0].barh(fa["feature_display"], fa["context_relevancy"], color=cols1, edgecolor="black")
axes[0, 0].axvline(ok["context_relevancy"].mean(), color="black", linestyle="--", linewidth=1)
axes[0, 0].set_xlabel("Context Relevancy")
axes[0, 0].set_title("Context Relevancy by Feature\n(red=high fabrication risk)", fontweight="bold")
axes[0, 0].tick_params(axis="y", labelsize=8)
axes[0, 0].set_xlim(0, 1)

# 2. Groundedness by feature
fa2 = feature_agg.sort_values("groundedness")
cols2 = [FAB_COLORS.get(r, "#95a5a6") for r in fa2["fabrication_risk"]]
axes[0, 1].barh(fa2["feature_display"], fa2["groundedness"], color=cols2, edgecolor="black")
axes[0, 1].axvline(ok["groundedness"].mean(), color="black", linestyle="--", linewidth=1)
axes[0, 1].set_xlabel("Groundedness")
axes[0, 1].set_title("Groundedness by Feature", fontweight="bold")
axes[0, 1].tick_params(axis="y", labelsize=8)
axes[0, 1].set_xlim(0, 1)

# 3. All 4 metrics — grouped bar by domain
dm = domain_agg.reset_index()
x  = np.arange(len(dm))
w  = 0.2
metric_colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
for i, (m, c) in enumerate(zip(METRIC_COLS, metric_colors)):
    axes[0, 2].bar(x + i * w, dm[m], w, label=m.replace("_", " ").title(), color=c, edgecolor="black")
axes[0, 2].set_xticks(x + w * 1.5)
axes[0, 2].set_xticklabels(dm["domain"])
axes[0, 2].set_title("HCAT Metrics by Domain", fontweight="bold")
axes[0, 2].legend(fontsize=8)
axes[0, 2].set_ylim(0, 1.05)

# 4. All 4 metrics by fabrication risk
fm = fab_agg.reset_index()
x2 = np.arange(len(fm))
for i, (m, c) in enumerate(zip(METRIC_COLS, metric_colors)):
    axes[1, 0].bar(x2 + i * w, fm[m], w, label=m.replace("_", " ").title(), color=c, edgecolor="black")
axes[1, 0].set_xticks(x2 + w * 1.5)
axes[1, 0].set_xticklabels(fm["fabrication_risk"])
axes[1, 0].set_title("HCAT Metrics by Fabrication Risk", fontweight="bold")
axes[1, 0].legend(fontsize=8)
axes[1, 0].set_ylim(0, 1.05)

# 5. Context relevancy distribution
for fab_risk, grp in ok.groupby("fabrication_risk"):
    axes[1, 1].hist(grp["context_relevancy"].dropna(), bins=20, alpha=0.6,
                    color=FAB_COLORS.get(fab_risk, "grey"), label=fab_risk, edgecolor="black")
axes[1, 1].set_xlabel("Context Relevancy")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Context Relevancy Distribution\nby Fabrication Risk", fontweight="bold")
axes[1, 1].legend(fontsize=9)

# 6. % cases requiring human review per feature
review_rate = (
    ok.groupby("feature_display")["requires_human_review"]
    .apply(lambda s: s.mean() * 100)
    .sort_values()
)
axes[1, 2].barh(review_rate.index, review_rate.values, color="#e67e22", edgecolor="black")
axes[1, 2].set_xlabel("% Cases Requiring Human Review")
axes[1, 2].set_title("Human Review Rate per Feature", fontweight="bold")
axes[1, 2].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hcat_embedding_summary.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {OUTPUT_DIR / 'hcat_embedding_summary.png'}")

---
## Part 6: Correlation with Validation Labels

Test whether low HCAT context quality predicts worse AI extraction outcomes.  
Join embedding metrics to validation sheet (AI annotation label) and compute:  
- Mean context relevancy stratified by AI label (1=correct, 2=omitted, 3=fabricated)  
- Spearman correlation between groundedness and AI correctness  
- ROC-AUC: can context_relevancy predict fabrication (ai_label == 3)?

In [ ]:
from scipy import stats
from sklearn.metrics import roc_auc_score

# Build a flat validation table with AI labels per feature key
AI_COL_MAP = {
    "lesion_size":                        "lesion_size_status_ai",
    "laterality":                         "laterality_status_ai",
    "lesion_location":                    "lesion_location_status_ai",
    "calcifications_asymmetry":           "calcifications_asymmetry_status_ai",
    "additional_enhancement_mri":         "additional_enhancement_mri_status_ai",
    "extent":                             "extent_status_ai",
    "accurate_clip_placement":            "accurate_clip_placement_status_ai",
    "workup_recommendation":              "workup_recommendation_status_ai",
    "lymph_node":                         "Lymph node_status_ai",
    "chronology_preserved":               "chronology_preserved_status_ai",
    "biopsy_method":                      "biopsy_method_status_ai",
    "invasive_component_size_pathology":  "invasive_component_size_pathology_status_ai",
    "histologic_diagnosis":               "histologic_diagnosis_status_ai",
    "receptor_status":                    "receptor_status_ai",
}

# Flatten validation sheet: one row per (obs_idx, feature_key)
val_long_rows = []
for obs_idx, row in val_df.iterrows():
    for fk, ai_col in AI_COL_MAP.items():
        if ai_col not in val_df.columns:
            continue
        ai_val = row[ai_col]
        if pd.isna(ai_val):
            continue
        try:
            ai_label = int(ai_val)
        except (ValueError, TypeError):
            continue
        val_long_rows.append({"obs_idx": obs_idx, "feature_key": fk, "ai_label": ai_label})

df_val_long = pd.DataFrame(val_long_rows)
print(f"Validation long format: {len(df_val_long)} rows")
print(f"AI label distribution: {df_val_long['ai_label'].value_counts().sort_index().to_dict()}")

In [ ]:
# Join embedding metrics to validation labels
# Aggregate per feature (context quality is case-level, labels are obs-level)
feature_context_means = (
    ok.groupby("feature_key")[METRIC_COLS].mean().reset_index()
)

val_feature_means = (
    df_val_long.groupby("feature_key")
    .agg(
        mean_ai_correct=("ai_label", lambda x: (x == 1).mean()),
        fab_rate=("ai_label", lambda x: (x == 3).mean()),
        omission_rate=("ai_label", lambda x: (x == 2).mean()),
        n_obs=("ai_label", "count")
    )
    .reset_index()
)

df_joined = feature_context_means.merge(val_feature_means, on="feature_key", how="inner")
print(f"Joined features: {len(df_joined)}")

print("\nSpearman correlations (context embedding metric → AI outcome):")
for emb_metric in METRIC_COLS:
    for outcome in ["mean_ai_correct", "fab_rate"]:
        if df_joined[emb_metric].std() > 0 and df_joined[outcome].std() > 0:
            rho, p = stats.spearmanr(df_joined[emb_metric], df_joined[outcome])
            sig = "*" if p < 0.05 else ""
            print(f"  {emb_metric:<25} vs {outcome:<20}: rho={rho:+.3f}  p={p:.3f} {sig}")

In [ ]:
# ── Scatter: context_relevancy vs fabrication rate per feature ─────────────────
if len(df_joined) >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Scatter: context_relevancy vs fab_rate
    ax = axes[0]
    sc = ax.scatter(
        df_joined["context_relevancy"], df_joined["fab_rate"],
        c=df_joined["mean_ai_correct"], cmap="RdYlGn",
        s=80, edgecolors="black", linewidths=0.5, vmin=0, vmax=1
    )
    for _, row in df_joined.iterrows():
        ax.annotate(
            FEATURE_DOCUMENT_CONTEXT.get(row["feature_key"], {}).get("display", row["feature_key"])[:15],
            (row["context_relevancy"], row["fab_rate"]),
            fontsize=6, xytext=(3, 3), textcoords="offset points"
        )
    plt.colorbar(sc, ax=ax, label="Mean AI Correct Rate")
    ax.set_xlabel("Mean Context Relevancy")
    ax.set_ylabel("Fabrication Rate (AI)")
    ax.set_title("Context Relevancy vs Fabrication Rate\n(per feature)", fontweight="bold")

    # Scatter: groundedness vs correctness
    ax2 = axes[1]
    sc2 = ax2.scatter(
        df_joined["groundedness"], df_joined["mean_ai_correct"],
        c=df_joined["fab_rate"], cmap="RdYlGn_r",
        s=80, edgecolors="black", linewidths=0.5, vmin=0, vmax=0.3
    )
    for _, row in df_joined.iterrows():
        ax2.annotate(
            FEATURE_DOCUMENT_CONTEXT.get(row["feature_key"], {}).get("display", row["feature_key"])[:15],
            (row["groundedness"], row["mean_ai_correct"]),
            fontsize=6, xytext=(3, 3), textcoords="offset points"
        )
    plt.colorbar(sc2, ax=ax2, label="Fabrication Rate")
    ax2.set_xlabel("Mean Groundedness")
    ax2.set_ylabel("Mean AI Correct Rate")
    ax2.set_title("Groundedness vs AI Correctness\n(per feature)", fontweight="bold")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "hcat_correlation_with_labels.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved → {OUTPUT_DIR / 'hcat_correlation_with_labels.png'}")
else:
    print("Not enough features with both embedding metrics and validation labels to scatter-plot.")

---
## Summary of Outputs

| Output | Path | Mode | Committed? |
|--------|------|------|------------|
| Context quality metrics | `data/processed/hcat_context_quality.csv` | Always | Yes |
| Answer quality metrics | `data/processed/hcat_answer_quality.csv` | After NB09 | Yes |
| HCAT metric plots | `reports/hcat_embedding_summary.png` | Always | Yes |
| Correlation with labels | `reports/hcat_correlation_with_labels.png` | Always | Yes |

**Key interpretive note:**  
In context-quality mode, `context_relevancy` measures how well the filtered source text matches  
the feature query — a proxy for whether the LLM will have sufficient information.  
Features with low `context_relevancy` + high `fabrication_risk` are your highest-priority cases  
for prompt engineering intervention in NB09.